# TP 2 — Comprendre `model.py` en profondeur — CORRIGÉ

## Objectif général

Ce corrigé explique le fonctionnement d'un GPT à partir de `model.py` de nanoGPT, en suivant les formes des tenseurs et leur rôle pédagogique.

Règle centrale : un LLM ne manipule pas du texte directement. Il manipule des tenseurs.  
Comprendre nanoGPT, c'est comprendre comment un tenseur d'indices `(B,T)` devient des logits `(B,T,vocab_size)`.


## Sources principales du parcours

Ces notebooks se basent principalement sur nanoGPT de Karpathy et sur la documentation officielle PyTorch/Jupyter.

- nanoGPT — README et quick start : https://github.com/karpathy/nanoGPT#quick-start  
  À lire pour comprendre le flux officiel : préparer Shakespeare, entraîner avec `train.py`, puis générer avec `sample.py`.
- nanoGPT — `model.py` : https://github.com/karpathy/nanoGPT/blob/master/model.py  
  À lire comme colonne vertébrale : `LayerNorm`, `CausalSelfAttention`, `MLP`, `Block`, `GPTConfig`, `GPT.forward`, `GPT.generate`.
- nanoGPT — `train.py` : https://github.com/karpathy/nanoGPT/blob/master/train.py  
  À lire pour la boucle d'entraînement : batch, forward, loss, backward, optimizer, checkpoint.
- nanoGPT — `sample.py` : https://github.com/karpathy/nanoGPT/blob/master/sample.py  
  À lire pour l'inférence : chargement du checkpoint, encodage du prompt, `model.generate`.
- Config Shakespeare caractère : https://github.com/karpathy/nanoGPT/blob/master/config/train_shakespeare_char.py  
  À lire pour comprendre les hyperparamètres pédagogiques : `block_size`, `n_layer`, `n_head`, `n_embd`, `dropout`, `max_iters`.
- PyTorch — Tensors : https://docs.pytorch.org/tutorials/beginner/basics/tensorqs_tutorial.html  
  À lire pour comprendre pourquoi les données, poids et activations sont des tenseurs.
- PyTorch — `torch.nn.Embedding` : https://docs.pytorch.org/docs/stable/generated/torch.nn.Embedding.html  
  À lire pour comprendre la table qui transforme des indices de tokens en vecteurs.
- PyTorch — `torch.nn.LayerNorm` : https://docs.pytorch.org/docs/stable/generated/torch.nn.LayerNorm.html  
  À lire pour comprendre la normalisation utilisée dans chaque bloc.
- PyTorch — `torch.nn.GELU` : https://docs.pytorch.org/docs/stable/generated/torch.nn.GELU.html  
  À lire pour comprendre l'activation du MLP.
- PyTorch — `torch.nn.CrossEntropyLoss` : https://docs.pytorch.org/docs/stable/generated/torch.nn.CrossEntropyLoss.html  
  À lire pour comprendre la loss de prédiction du prochain token.
- PyTorch — `torch.nn.functional.scaled_dot_product_attention` : https://docs.pytorch.org/docs/stable/generated/torch.nn.functional.scaled_dot_product_attention.html  
  À lire pour comprendre la version optimisée de l'attention Q/K/V.
- PyTorch — CUDA semantics : https://docs.pytorch.org/docs/stable/notes/cuda.html  
  À lire pour comprendre le placement CPU/GPU des tenseurs et modèles.
- Jupyter — Markdown cells : https://jupyter-notebook.readthedocs.io/en/stable/examples/Notebook/Working%20With%20Markdown%20Cells.html  
  À lire pour savoir structurer les réponses dans les cellules Markdown.


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

torch.manual_seed(42)
print("PyTorch:", torch.__version__)


## Partie 1 — Les dimensions fondamentales

### Question 2.1 — Comprendre les dimensions `(B, T, C)`

**Solution :**

- `B` signifie **batch size** : nombre de séquences traitées en parallèle.
- `T` signifie **time** ou longueur de contexte : nombre de tokens dans chaque séquence.
- `C` signifie **channels** ou dimension d'embedding : taille du vecteur qui représente chaque token.

Donc un tenseur `(B,T,C)` signifie :

> pour chaque exemple du batch, pour chaque position de la séquence, on stocke un vecteur de taille `C`.


In [ ]:
B, T, C = 2, 4, 8
x = torch.randn(B, T, C)

print("x.shape =", x.shape)
print("B =", x.shape[0], "séquences dans le batch")
print("T =", x.shape[1], "tokens par séquence")
print("C =", x.shape[2], "valeurs par vecteur/token")


### Explication pédagogique détaillée

Imagine un tableau à trois niveaux :

1. Niveau batch : plusieurs phrases sont traitées en même temps.
2. Niveau temps : chaque phrase est découpée en positions successives.
3. Niveau embedding : chaque position est représentée par un vecteur numérique.

Dans un LLM, cette structure permet de paralléliser les calculs. Au lieu de traiter une phrase puis une autre, on traite plusieurs séquences simultanément.

**À quoi ça sert dans nanoGPT ?**  
Dans `GPT.forward`, `idx` commence avec la forme `(B,T)`, car ce sont seulement des identifiants de tokens. Après embedding, on obtient `(B,T,C)`, car chaque token devient un vecteur de taille `C`.

**Erreur fréquente :**  
Ne confonds pas `T` et `C`.  
`T` est le nombre de positions.  
`C` est la taille du vecteur à chaque position.


### Question 2.2 — Comprendre `idx` et `targets`

**Solution :**

- `idx` contient les tokens donnés au modèle en entrée.
- `targets` contient les tokens que le modèle doit prédire.
- En entraînement autoregressif, `targets` est généralement la même séquence décalée d'un cran vers la droite.


In [ ]:
sequence = torch.tensor([0, 1, 2, 3, 4, 5])

# On veut prédire le token suivant.
idx = sequence[:-1]
targets = sequence[1:]

print("sequence:", sequence.tolist())
print("idx    :", idx.tolist())
print("targets:", targets.tolist())

# Exemple : quand le modèle voit 0, il doit prédire 1.
for x_i, y_i in zip(idx.tolist(), targets.tolist()):
    print(f"entrée {x_i} -> cible {y_i}")


### Explication pédagogique détaillée

Un GPT est un modèle **autoregressif** : il apprend à prédire le prochain token à partir des tokens précédents.

Exemple :

```text
texte :     A B C D
entrée :    A B C
cible :     B C D
```

À chaque position, le modèle voit le passé et doit deviner le token suivant.

Dans `train.py`, nanoGPT construit les batches ainsi :
- `x = data[i : i + block_size]`
- `y = data[i+1 : i+1 + block_size]`

C'est ce décalage qui transforme un simple texte en tâche supervisée.

**À quoi ça sert dans un LLM ?**  
C'est le principe de base de l'entraînement des GPT : apprendre la distribution du prochain token.

**Erreur fréquente :**  
Penser que `targets` est une étiquette globale de phrase. Non : il y a une cible par position.


## Partie 2 — `GPTConfig` et les hyperparamètres

### Question 2.3 — Lire `GPTConfig`

**Solution :**

| Hyperparamètre | Rôle |
|---|---|
| `block_size` | Longueur maximale du contexte, donc valeur maximale de `T`. |
| `vocab_size` | Nombre de tokens possibles dans le vocabulaire. |
| `n_layer` | Nombre de blocs Transformer empilés. |
| `n_head` | Nombre de têtes d'attention par bloc. |
| `n_embd` | Dimension des vecteurs internes, donc valeur de `C`. |
| `dropout` | Régularisation : met aléatoirement certaines activations à zéro pendant l'entraînement. |
| `bias` | Active ou désactive certains biais dans les couches `Linear` et `LayerNorm`. |


### Explication pédagogique détaillée

Ces valeurs définissent la capacité du modèle.

- Plus `n_layer` est grand, plus le modèle peut empiler de transformations.
- Plus `n_embd` est grand, plus chaque token a une représentation riche.
- Plus `n_head` est grand, plus l'attention peut regarder la séquence sous plusieurs sous-espaces.
- Plus `block_size` est grand, plus le modèle peut utiliser un contexte long, mais plus l'attention coûte cher.

**Lien avec la mémoire GPU :**  
L'attention naïve crée des matrices de forme `(B, n_head, T, T)`.  
Donc si tu doubles `T`, la mémoire de l'attention peut augmenter environ comme `T²`.

**Erreur fréquente :**  
Croire que `vocab_size` est la longueur de la séquence. Non.  
`vocab_size` est le nombre de symboles possibles ; `block_size` est la longueur du contexte.


### Question 2.4 — Calculer `head_size`

**Solution :**


In [ ]:
n_embd = 8
n_head = 2

assert n_embd % n_head == 0
head_size = n_embd // n_head

print("n_embd =", n_embd)
print("n_head =", n_head)
print("head_size =", head_size)


### Explication pédagogique détaillée

Le multi-head attention découpe la dimension totale `C` en plusieurs têtes.

Si `C = 8` et `n_head = 2`, alors chaque tête travaille sur `4` dimensions.

```text
C = 8
head 1 : dimensions 0,1,2,3
head 2 : dimensions 4,5,6,7
```

Dans nanoGPT, chaque tête produit une attention différente. Ensuite, les têtes sont recombinées.

**À quoi ça sert ?**  
Une seule tête pourrait apprendre un type de relation. Plusieurs têtes peuvent apprendre plusieurs types de relations en parallèle : proximité locale, dépendance syntaxique, répétition, ponctuation, etc.

**Erreur fréquente :**  
Penser que chaque tête voit toute la dimension `C`. Dans cette implémentation, chaque tête reçoit `C / n_head` dimensions.


## Partie 3 — Embeddings : passer des tokens aux vecteurs

### Question 2.5 — Comprendre les token embeddings

**Solution :**


In [ ]:
B, T, C, vocab_size = 2, 4, 8, 16

idx = torch.randint(0, vocab_size, (B, T))
wte = nn.Embedding(vocab_size, C)
tok_emb = wte(idx)

print("idx:")
print(idx)
print("idx.shape     =", idx.shape)
print("tok_emb.shape =", tok_emb.shape)


### Explication pédagogique détaillée

`idx` contient des entiers.  
Ces entiers sont des identifiants de tokens, pas des vecteurs de sens.

`nn.Embedding(vocab_size, C)` est une table de taille :

```text
vocab_size lignes × C colonnes
```

Quand on fait `wte(idx)`, PyTorch va chercher les lignes correspondant aux indices.

Si `idx[b,t] = 5`, alors `tok_emb[b,t]` reçoit la ligne 5 de la table d'embedding.

**À quoi ça sert dans un LLM ?**  
Le modèle ne peut pas apprendre sur des caractères ou mots bruts. Il a besoin d'une représentation continue.  
L'embedding est le premier espace vectoriel du modèle.

**Lien direct avec nanoGPT :**

Dans `model.py` :
- `wte = nn.Embedding(config.vocab_size, config.n_embd)`
- `tok_emb = self.transformer.wte(idx)`

**Erreur fréquente :**  
Penser que le numéro du token a une signification numérique. Le token 10 n'est pas deux fois le token 5. Le sens est dans la ligne d'embedding apprise.


### Question 2.6 — Comprendre les position embeddings

**Solution :**


In [ ]:
T, C, block_size = 4, 8, 16

pos = torch.arange(0, T, dtype=torch.long)
wpe = nn.Embedding(block_size, C)
pos_emb = wpe(pos)

print("pos:", pos)
print("pos.shape     =", pos.shape)
print("pos_emb.shape =", pos_emb.shape)


### Explication pédagogique détaillée

L'attention compare les tokens entre eux, mais elle ne contient pas naturellement l'ordre de la séquence.

Exemple :

```text
chien mord homme
homme mord chien
```

Ces deux séquences contiennent les mêmes tokens, mais l'ordre change complètement le sens.

Le position embedding ajoute une information de position à chaque token :
- position 0 ;
- position 1 ;
- position 2 ;
- etc.

Dans nanoGPT, `wpe` est une table qui associe chaque position possible à un vecteur.

**Pourquoi `pos_emb` a la forme `(T,C)` ?**  
Il y a un vecteur de position pour chaque position de la séquence, pas pour chaque élément du batch. Le même vecteur de position est utilisé pour tous les exemples du batch.

**Erreur fréquente :**  
Croire que le Transformer connaît naturellement l'ordre. Sans information de position, il serait beaucoup plus difficile de distinguer l'ordre des tokens.


### Question 2.7 — Additionner token et position embeddings

**Solution :**


In [ ]:
B, T, C = 2, 4, 8

tok_emb = torch.randn(B, T, C)
pos_emb = torch.randn(T, C)

x = tok_emb + pos_emb

print("tok_emb.shape =", tok_emb.shape)
print("pos_emb.shape =", pos_emb.shape)
print("x.shape       =", x.shape)


### Explication pédagogique détaillée

`tok_emb` a la forme `(B,T,C)` : un vecteur par token dans chaque séquence.  
`pos_emb` a la forme `(T,C)` : un vecteur par position.

PyTorch applique le **broadcasting** : il ajoute le même `pos_emb` à chaque élément du batch.

On obtient donc :

```text
x[b,t,:] = token_embedding_du_token[b,t] + position_embedding_de_t
```

**À quoi ça sert ?**  
`x` contient à la fois :
- l'identité du token ;
- sa position dans la séquence.

C'est cette représentation qui entre dans les blocs Transformer.

**Erreur fréquente :**  
Penser que les embeddings sont concaténés. Dans nanoGPT, ils sont additionnés, pas concaténés.


## Partie 4 — Entrer dans un bloc Transformer

### Question 2.8 — Lire la structure de `Block`

**Solution :**

Dans nanoGPT, un bloc fait essentiellement :

```python
x = x + self.attn(self.ln_1(x))
x = x + self.mlp(self.ln_2(x))
```

Cela signifie :

1. normaliser `x` ;
2. appliquer l'attention ;
3. ajouter le résultat à `x` via une connexion résiduelle ;
4. normaliser à nouveau ;
5. appliquer le MLP ;
6. ajouter le résultat à `x` via une deuxième connexion résiduelle.


### Explication pédagogique détaillée

Un bloc Transformer GPT conserve généralement la forme `(B,T,C)`.  
Il change le contenu des vecteurs, pas leur structure.

La connexion résiduelle `x + transformation(x)` sert à garder une route directe pour l'information et les gradients.  
Sans résidus, les réseaux très profonds sont plus difficiles à entraîner.

La version de nanoGPT est dite **Pre-LN** : la normalisation arrive avant l'attention et avant le MLP.

**À quoi ça sert dans un LLM ?**  
Chaque bloc enrichit la représentation de chaque token :
- l'attention permet aux positions de lire les autres positions passées ;
- le MLP transforme chaque position individuellement ;
- les résidus évitent de perdre l'information déjà présente.

**Erreur fréquente :**  
Penser que l'attention réduit la séquence. Non : elle renvoie encore un vecteur pour chaque position.


### Question 2.9 — Comprendre LayerNorm

**Solution :**


In [ ]:
B, T, C = 2, 4, 8

x = torch.randn(B, T, C)
ln = nn.LayerNorm(C)
y = ln(x)

print("x.shape =", x.shape)
print("y.shape =", y.shape)

# Vérifions sur le dernier axe : chaque vecteur de taille C est normalisé.
print("Moyenne approx sur C pour le premier token :", y[0, 0].mean().item())
print("Écart-type approx sur C pour le premier token :", y[0, 0].std(unbiased=False).item())


### Explication pédagogique détaillée

LayerNorm normalise chaque vecteur de taille `C` séparément.

Pour un tenseur `(B,T,C)`, elle prend chaque vecteur `x[b,t,:]`, calcule sa moyenne et sa variance, puis le remet à une échelle plus stable.

Cela ne mélange pas les exemples du batch.  
Cela ne mélange pas les positions.  
Cela agit sur la dernière dimension : la dimension d'embedding.

**À quoi ça sert ?**  
Les valeurs internes d'un réseau profond peuvent devenir trop grandes, trop petites ou instables. LayerNorm stabilise les activations, ce qui rend l'entraînement plus robuste.

**Lien avec nanoGPT :**  
Chaque bloc a :
- `ln_1` avant l'attention ;
- `ln_2` avant le MLP ;
- une `ln_f` finale avant la projection vers le vocabulaire.

**Erreur fréquente :**  
Confondre LayerNorm et BatchNorm. LayerNorm ne dépend pas de la taille du batch de la même manière que BatchNorm.


## Partie 5 — Attention causale pas à pas

### Question 2.10 — Projeter `x` en Q, K, V

**Solution :**


In [ ]:
B, T, C = 2, 4, 8

x = torch.randn(B, T, C)
c_attn = nn.Linear(C, 3*C)

qkv = c_attn(x)

print("x.shape   =", x.shape)
print("qkv.shape =", qkv.shape)


### Explication pédagogique détaillée

L'attention a besoin de trois représentations :

- **Query (Q)** : ce que la position courante cherche.
- **Key (K)** : ce que chaque position propose comme clé de comparaison.
- **Value (V)** : l'information réellement récupérée si une position est pertinente.

nanoGPT calcule les trois en une seule couche linéaire :

```text
(B,T,C) → (B,T,3C)
```

Puis il découpe ce résultat en trois morceaux de taille `C`.

**Pourquoi une seule Linear ?**  
C'est plus compact et souvent plus efficace que trois Linear séparées. Mathématiquement, c'est équivalent à avoir trois projections regroupées.

**À quoi ça sert dans un LLM ?**  
Q et K déterminent *qui regarde qui*.  
V détermine *quelle information est transmise*.

**Erreur fréquente :**  
Penser que Q, K, V sont des copies de `x`. Non : ce sont des projections apprises, donc des transformations différentes de `x`.


### Question 2.11 — Découper Q, K, V

**Solution :**


In [ ]:
B, T, C = 2, 4, 8

qkv = torch.randn(B, T, 3*C)
q, k, v = qkv.split(C, dim=2)

print("qkv.shape =", qkv.shape)
print("q.shape   =", q.shape)
print("k.shape   =", k.shape)
print("v.shape   =", v.shape)


### Explication pédagogique détaillée

La dimension `2` correspond à la dimension des features.

Comme `qkv` a `3C` features, on le découpe en trois tenseurs de `C` features :

```text
qkv : (B,T,3C)
q   : (B,T,C)
k   : (B,T,C)
v   : (B,T,C)
```

**À quoi ça sert ?**  
On a maintenant les trois ingrédients de l'attention.  
La suite consiste à les réorganiser pour que plusieurs têtes puissent travailler en parallèle.

**Erreur fréquente :**  
Découper sur la mauvaise dimension. Il faut découper sur la dimension des canaux, pas sur le batch ni sur le temps.


### Question 2.12 — Préparer le multi-head attention

**Solution :**


In [ ]:
B, T, C, n_head = 2, 4, 8, 2
head_size = C // n_head

q = torch.randn(B, T, C)

q_heads = q.view(B, T, n_head, head_size).transpose(1, 2)

print("q.shape       =", q.shape)
print("head_size     =", head_size)
print("q_heads.shape =", q_heads.shape)


### Explication pédagogique détaillée

On part de :

```text
q : (B,T,C)
```

On sait que :

```text
C = n_head × head_size
```

Donc on peut réécrire la dernière dimension :

```text
(B,T,C) → (B,T,n_head,head_size)
```

Puis on transpose pour obtenir :

```text
(B,n_head,T,head_size)
```

Cette forme est pratique car chaque tête a sa propre matrice d'attention sur les `T` positions.

**À quoi ça sert ?**  
Au lieu d'une seule attention, le modèle calcule plusieurs attentions en parallèle. Chaque tête peut apprendre une manière différente de regarder le contexte.

**Erreur fréquente :**  
Oublier que `view` ne change pas les valeurs, seulement leur organisation logique. La projection a déjà été faite par `c_attn`.


### Question 2.13 — Calculer les scores d'attention

**Solution :**


In [ ]:
B, n_head, T, head_size = 2, 2, 4, 4

q = torch.randn(B, n_head, T, head_size)
k = torch.randn(B, n_head, T, head_size)

scores = q @ k.transpose(-2, -1)
scores_scaled = scores / math.sqrt(head_size)

print("q.shape             =", q.shape)
print("k.transpose shape   =", k.transpose(-2, -1).shape)
print("scores.shape        =", scores.shape)
print("scores_scaled.shape =", scores_scaled.shape)


### Explication pédagogique détaillée

Chaque position possède une query.  
Chaque position possède aussi une key.

Le produit `q @ k.transpose(-2, -1)` compare chaque query avec toutes les keys.

Pour chaque batch et chaque tête, on obtient une matrice :

```text
(T,T)
```

La valeur `scores[b,h,i,j]` signifie :

> dans le batch `b`, pour la tête `h`, à quel point la position `i` doit-elle regarder la position `j` ?

La division par `sqrt(head_size)` stabilise les scores. Sans cette mise à l'échelle, les produits scalaires peuvent devenir grands, ce qui rend le softmax trop extrême.

**À quoi ça sert dans un LLM ?**  
C'est là que le modèle décide quelles positions passées sont utiles pour prédire la suite.

**Erreur fréquente :**  
Croire que la matrice `(T,T)` est une matrice de tokens. Non, c'est une matrice de relations entre positions.


### Question 2.14 — Appliquer le masque causal

**Solution :**


In [ ]:
T = 4

# Matrice triangulaire inférieure : True là où l'attention est autorisée.
mask = torch.tril(torch.ones(T, T, dtype=torch.bool))

print(mask)

scores = torch.randn(1, 1, T, T)
masked_scores = scores.masked_fill(mask == 0, float("-inf"))

print("Scores avant masque:")
print(scores[0, 0])
print("Scores après masque:")
print(masked_scores[0, 0])


### Explication pédagogique détaillée

GPT génère de gauche à droite.  
À la position `i`, il ne doit pas voir les positions futures `j > i`.

Pour `T=4`, le masque autorise :

```text
position 0 voit : 0
position 1 voit : 0, 1
position 2 voit : 0, 1, 2
position 3 voit : 0, 1, 2, 3
```

Les positions futures sont remplacées par `-inf`.  
Après softmax, `exp(-inf) = 0`, donc ces positions reçoivent une probabilité nulle.

**À quoi ça sert dans un LLM ?**  
Sans masque causal, le modèle tricherait pendant l'entraînement : il pourrait regarder la réponse future.  
Le masque force le modèle à apprendre la vraie tâche autoregressive.

**Erreur fréquente :**  
Penser que le masque sert seulement à l'inférence. Il est crucial aussi pendant l'entraînement.


### Question 2.15 — Transformer les scores en probabilités

**Solution :**


In [ ]:
T = 4
scores = torch.randn(1, 1, T, T)

mask = torch.tril(torch.ones(T, T, dtype=torch.bool))
masked_scores = scores.masked_fill(mask == 0, float("-inf"))

att = F.softmax(masked_scores, dim=-1)

print("Attention:")
print(att[0, 0])
print("Sommes sur la dernière dimension:")
print(att.sum(dim=-1))


### Explication pédagogique détaillée

Le softmax transforme les scores en poids positifs qui somment à 1.

Pour une position donnée `i`, la ligne `att[..., i, :]` répond à la question :

> quelle proportion d'attention la position `i` donne-t-elle à chaque position `j` autorisée ?

`dim=-1` signifie qu'on normalise sur les positions regardées.

**À quoi ça sert ?**  
Ces poids vont servir à faire une moyenne pondérée des values `v`.

**Erreur fréquente :**  
Dire que le softmax choisit une seule position. Non : il produit une distribution. Plusieurs positions peuvent recevoir du poids.


### Question 2.16 — Calculer `att @ v`

**Solution :**


In [ ]:
B, n_head, T, head_size = 2, 2, 4, 4

att = torch.randn(B, n_head, T, T).softmax(dim=-1)
v = torch.randn(B, n_head, T, head_size)

y = att @ v

print("att.shape =", att.shape)
print("v.shape   =", v.shape)
print("y.shape   =", y.shape)


### Explication pédagogique détaillée

`att` contient les poids d'attention.  
`v` contient les informations à récupérer.

Pour chaque position `i`, on calcule une combinaison pondérée :

```text
y[i] = somme_j att[i,j] * v[j]
```

Donc `y[b,h,i]` est le nouveau vecteur de la position `i`, pour la tête `h`, après avoir lu les positions autorisées.

**À quoi ça sert dans un LLM ?**  
C'est le moment où l'information circule entre tokens.  
Avant cela, chaque position avait son vecteur. Après cela, chaque position contient un mélange d'informations du contexte passé.

**Erreur fréquente :**  
Penser que Q/K/V ont tous le même rôle. Q et K servent à calculer les poids ; V transporte le contenu.


### Question 2.17 — Recombiner les têtes

**Solution :**


In [ ]:
B, T, C, n_head = 2, 4, 8, 2
head_size = C // n_head

y = torch.randn(B, n_head, T, head_size)
y_recombined = y.transpose(1, 2).contiguous().view(B, T, C)

print("y.shape            =", y.shape)
print("y_recombined.shape =", y_recombined.shape)


### Explication pédagogique détaillée

Les têtes ont travaillé séparément :

```text
(B,n_head,T,head_size)
```

Mais le reste du modèle attend un vecteur unique de taille `C` par position :

```text
(B,T,C)
```

On transpose donc pour revenir à :

```text
(B,T,n_head,head_size)
```

puis on fusionne `n_head × head_size` en `C`.

`.contiguous()` est utilisé après `transpose`, car le tenseur peut ne plus être stocké de façon contiguë en mémoire. `view` a souvent besoin d'une mémoire organisée de manière compatible.

**À quoi ça sert ?**  
Après recombinaison, on peut appliquer la projection `c_proj`, puis continuer le bloc Transformer.

**Erreur fréquente :**  
Oublier que les têtes sont une organisation interne temporaire. L'interface du bloc reste `(B,T,C)`.


## Partie 6 — MLP, logits et loss

### Question 2.18 — Comprendre le MLP

**Solution :**


In [ ]:
B, T, C = 2, 4, 8

mlp = nn.Sequential(
    nn.Linear(C, 4*C),
    nn.GELU(),
    nn.Linear(4*C, C),
)

x = torch.randn(B, T, C)
y = mlp(x)

print("x.shape =", x.shape)
print("y.shape =", y.shape)


### Explication pédagogique détaillée

Le MLP transforme chaque position indépendamment.

La structure est :

```text
C → 4C → C
```

On agrandit temporairement la dimension pour donner plus de capacité au réseau, puis on revient à `C` pour rester compatible avec les connexions résiduelles et les blocs suivants.

GELU ajoute une non-linéarité. Sans activation non linéaire, empiler des couches linéaires reviendrait essentiellement à une seule grande transformation linéaire.

**Rôle complémentaire avec l'attention :**
- Attention : mélange l'information entre positions.
- MLP : transforme l'information à l'intérieur de chaque position.

**Erreur fréquente :**  
Croire que le MLP regarde les autres tokens. Dans cette implémentation, le MLP agit position par position.


### Question 2.19 — Comprendre les logits

**Solution :**


In [ ]:
B, T, C, vocab_size = 2, 4, 8, 16

x = torch.randn(B, T, C)
lm_head = nn.Linear(C, vocab_size, bias=False)

logits = lm_head(x)

print("x.shape      =", x.shape)
print("logits.shape =", logits.shape)


### Explication pédagogique détaillée

À la fin du modèle, chaque position possède un vecteur de taille `C`.  
Mais pour prédire un token, il faut produire un score pour chaque token possible du vocabulaire.

`lm_head` projette donc :

```text
C → vocab_size
```

La sortie s'appelle `logits`.

Un logit est un score brut, pas une probabilité.  
Pour obtenir des probabilités, on appliquerait un softmax.

**À quoi ça sert dans un LLM ?**  
À chaque position, le modèle donne un score à tous les prochains tokens possibles. Le token cible doit recevoir un score plus élevé que les autres.

**Erreur fréquente :**  
Penser que le plus grand logit est toujours choisi pendant l'inférence. En sampling, on peut échantillonner selon la distribution, avec `temperature` et `top_k`.


### Question 2.20 — Comprendre la cross-entropy loss

**Solution :**


In [ ]:
B, T, vocab_size = 2, 4, 16

logits = torch.randn(B, T, vocab_size)
targets = torch.randint(0, vocab_size, (B, T))

loss = F.cross_entropy(
    logits.view(-1, logits.size(-1)),
    targets.view(-1)
)

print("logits avant flatten :", logits.shape)
print("targets avant flatten:", targets.shape)
print("logits après flatten :", logits.view(-1, logits.size(-1)).shape)
print("targets après flatten:", targets.view(-1).shape)
print("loss:", loss.item())


### Explication pédagogique détaillée

La cross-entropy attend :
- une matrice de prédictions `(N, vocab_size)` ;
- un vecteur de classes cibles `(N)`.

Mais nanoGPT produit :
- `logits` de forme `(B,T,vocab_size)` ;
- `targets` de forme `(B,T)`.

On a donc `N = B*T` prédictions.  
On aplatit les deux premières dimensions :

```text
(B,T,V) → (B*T,V)
(B,T)   → (B*T)
```

La loss compare ensuite chaque ligne de logits avec la classe cible correspondante.

**À quoi ça sert dans un LLM ?**  
Cela entraîne le modèle à augmenter le score du vrai prochain token à chaque position.

**Erreur fréquente :**  
Appliquer softmax avant `cross_entropy`. En PyTorch, `F.cross_entropy` attend des logits bruts et applique la logique nécessaire en interne.


## Partie 7 — Synthèse

### Question 2.21 — Raconter le forward pass complet

**Solution synthèse :**

1. Le modèle reçoit `idx` de forme `(B,T)`, qui contient des identifiants de tokens.
2. `wte(idx)` transforme chaque identifiant en vecteur : on obtient `tok_emb` de forme `(B,T,C)`.
3. `wpe(pos)` crée les embeddings de position : `pos_emb` de forme `(T,C)`.
4. `tok_emb + pos_emb` produit `x`, qui contient l'identité des tokens et leur position.
5. `x` traverse plusieurs blocs Transformer.
6. Dans chaque bloc :
   - LayerNorm stabilise les activations ;
   - l'attention causale permet à chaque position de lire les positions précédentes ;
   - une connexion résiduelle préserve l'information ;
   - un MLP transforme chaque position ;
   - une deuxième connexion résiduelle préserve encore l'information.
7. Après tous les blocs, une LayerNorm finale produit une représentation stable.
8. `lm_head(x)` projette chaque vecteur de taille `C` vers `vocab_size` logits.
9. Si `targets` est fourni, `F.cross_entropy` compare les logits aux vrais prochains tokens.
10. Le résultat est une loss scalaire utilisée pour `backward()` pendant l'entraînement.

**Phrase à retenir :**

> Un GPT apprend à transformer une séquence de tokens en une distribution de probabilité du prochain token, position par position, en utilisant des blocs d'attention causale et des MLP.


In [ ]:
# Mini-forward pass récapitulatif, sans classe complète GPT.
B, T, C, vocab_size, n_head = 2, 4, 8, 16, 2
head_size = C // n_head

idx = torch.randint(0, vocab_size, (B, T))
targets = torch.randint(0, vocab_size, (B, T))

wte = nn.Embedding(vocab_size, C)
wpe = nn.Embedding(T, C)
ln = nn.LayerNorm(C)
c_attn = nn.Linear(C, 3*C)
c_proj = nn.Linear(C, C)
mlp = nn.Sequential(nn.Linear(C, 4*C), nn.GELU(), nn.Linear(4*C, C))
lm_head = nn.Linear(C, vocab_size, bias=False)

pos = torch.arange(0, T)
x = wte(idx) + wpe(pos)

# attention simplifiée
x_norm = ln(x)
q, k, v = c_attn(x_norm).split(C, dim=2)
q = q.view(B, T, n_head, head_size).transpose(1, 2)
k = k.view(B, T, n_head, head_size).transpose(1, 2)
v = v.view(B, T, n_head, head_size).transpose(1, 2)

scores = (q @ k.transpose(-2, -1)) / math.sqrt(head_size)
mask = torch.tril(torch.ones(T, T, dtype=torch.bool))
scores = scores.masked_fill(mask == 0, float("-inf"))
att = F.softmax(scores, dim=-1)
y = att @ v
y = y.transpose(1, 2).contiguous().view(B, T, C)
x = x + c_proj(y)

# mlp
x = x + mlp(ln(x))

logits = lm_head(ln(x))
loss = F.cross_entropy(logits.view(-1, vocab_size), targets.view(-1))

print("idx    :", idx.shape)
print("x      :", x.shape)
print("logits :", logits.shape)
print("loss   :", loss.item())
